# Given the following text, generate word-level text using both RNN and LSTM models. Use AraVec V2 to obtain the word embeddings.

مُنذ زمن بعيد كانت هناك قرية قديمة مهجورة، حيث كانت مليئة بالمنازل والشوارع والمحلات التجاريّة القديمة الفارغة، ممّا جعل هذه القرية مكانًا جيِّدًا لتعيش فيه الفئران!

كانت الفئران تعيش بسعادة في هذه المنطقة لمئات السنين، حتّى قبل أن يأتي الناس لبناء القرية ثم يغادرون. ولكن بعد أن غادر الناس عاش الفئران أفضل أوقاتهم، وصنعوا أنفاقًا عبر المنازل والمباني القديمة المهجورة، ليتحرّكوا من خلالها بحريّة دون وجود أي خطر عليهم.

In [1]:
import numpy as np
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


In [2]:
text = """
مُنذ زمن بعيد كانت هناك قرية قديمة مهجورة، حيث كانت مليئة بالمنازل والشوارع والمحلات التجاريّة القديمة الفارغة، ممّا جعل هذه القرية مكانًا جيِّدًا لتعيش فيه الفئران!
كانت الفئران تعيش بسعادة في هذه المنطقة لمئات السنين، حتّى قبل أن يأتي الناس لبناء القرية ثم يغادرون. ولكن بعد أن غادر الناس عاش الفئران أفضل أوقاتهم، وصنعوا أنفاقًا عبر المنازل والمباني القديمة المهجورة، ليتحرّكوا من خلالها بحريّة دون وجود أي خطر عليهم.
"""


In [3]:
# 2. TOKENIZATION
# ─────────────────────────────────────────────────────────────────────────────
words    = text.strip().split()
vocab    = sorted(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)   # 60 unique words

print(f"Vocabulary size : {vocab_size}")
print(f"Total tokens    : {len(words)}")

Vocabulary size : 61
Total tokens    : 69


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hadeerkhalednabil/full-grams-cbow-100-twitter")

print("Path to dataset files:", path)

100%|██████████| 1.05G/1.05G [00:08<00:00, 134MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hadeerkhalednabil/full-grams-cbow-100-twitter/versions/2


In [12]:
import gensim
import os

# Load the AraVec model (full_grams_cbow_100_twitter.mdl)
# Correcting the path to use the '.mdl' extension
model_path = os.path.join(path, 'full_grams_cbow_100_twitter.mdl')
ARAVEC_MODEL = gensim.models.Word2Vec.load(model_path)

print(f"AraVec model loaded successfully from: {model_path}")
print(f"Vector size: {ARAVEC_MODEL.vector_size}")

AraVec model loaded successfully from: /root/.cache/kagglehub/datasets/hadeerkhalednabil/full-grams-cbow-100-twitter/versions/2/full_grams_cbow_100_twitter.mdl
Vector size: 100


In [13]:
# 3. EMBEDDING LAYER
# ─────────────────────────────────────────────────────────────────────────────
def get_word_embedding(word):
    try:
        return ARAVEC_MODEL.wv[word]
    except KeyError:
        # Return a zero vector for words not found in the AraVec model
        return np.zeros(ARAVEC_MODEL.vector_size)

# Create an embedding matrix for our vocabulary
embedding_dim = ARAVEC_MODEL.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word2idx.items():
    embedding_matrix[i] = get_word_embedding(word)

print(f"Embedding matrix shape: {embedding_matrix.shape}")

Embedding matrix shape: (61, 100)


In [15]:
 #SEQUENCE PREPARATION
 #─────────────────────────────────────────────────────────────────────────────

SEQ_LEN = 5 # Define the sequence length

from tensorflow.keras.utils import to_categorical

def make_sequences(words, seq_len):
    indices = [word2idx[w] for w in words]
    X = [indices[i:i+seq_len]        for i in range(len(indices)-seq_len)]
    y = [indices[i+seq_len]          for i in range(len(indices)-seq_len)]
    return np.array(X), np.array(y)

X, y   = make_sequences(words, SEQ_LEN)
y_cat  = to_categorical(y, num_classes=vocab_size)
print(f"Training samples: {len(X)}  (seq_len={SEQ_LEN})")

Training samples: 64  (seq_len=5)


In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout

# 4. RNN MODEL
# ─────────────────────────────────────────────────────────────────────────────
rnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=SEQ_LEN, trainable=False),
    SimpleRNN(128, return_sequences=True),
    Dropout(0.2),
    SimpleRNN(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │         6,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,100 (23.83 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,100 (23.83 KB)

In [17]:
# 5. LSTM MODEL
# ─────────────────────────────────────────────────────────────────────────────
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=SEQ_LEN, trainable=False),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │         6,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,100 (23.83 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,100 (23.83 KB)

In [18]:
# 6. TRAIN RNN MODEL
# ─────────────────────────────────────────────────────────────────────────────
rnn_history = rnn_model.fit(X, y_cat, epochs=50, verbose=1)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.0000e+00 - loss: 4.4785
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0625 - loss: 3.9132
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.2188 - loss: 3.5312
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2812 - loss: 3.2364
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.4219 - loss: 2.9129
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5781 - loss: 2.6173
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5781 - loss: 2.4619
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7969 - loss: 2.1052
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8594 - loss: 1.9207
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8906 - loss: 1.7904
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9688 - loss: 1.5265
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9375 - loss: 1.46

In [19]:
# 7. TRAIN LSTM MODEL
# ─────────────────────────────────────────────────────────────────────────────
lstm_history = lstm_model.fit(X, y_cat, epochs=50, verbose=1)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.0000e+00 - loss: 4.1190
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.0938 - loss: 4.0391
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.1094 - loss: 3.9839
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.2188 - loss: 3.9235
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.2188 - loss: 3.8474
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.2969 - loss: 3.7732
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4688 - loss: 3.6602
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.3125 - loss: 3.5663
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.2969 - loss: 3.4339
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3594 - loss: 3.2878
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3594 - loss: 3.1671
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4531 - loss: 2.93

In [26]:
# 8. TEXT GENERATION
# ─────────────────────────────────────────────────────────────────────────────
def generate_text(model, seed_text, num_gen_words, seq_len):
    generated_text = []
    # Ensure seed_text words exactly match vocabulary entries, including punctuation.
    # The vocabulary was built using text.strip().split(), which preserves punctuation.
    input_sequence = [word2idx[word] for word in seed_text.split()]

    for _ in range(num_gen_words):
        # Pad sequence if its length is less than seq_len
        if len(input_sequence) < seq_len:
            padded_input_sequence = [0] * (seq_len - len(input_sequence)) + input_sequence
        else:
            padded_input_sequence = input_sequence[-seq_len:]

        # Predict next word
        predicted_probs = model.predict(np.array(padded_input_sequence).reshape(1, seq_len), verbose=0)[0]
        predicted_word_idx = np.argmax(predicted_probs)
        predicted_word = idx2word[predicted_word_idx]

        generated_text.append(predicted_word)
        input_sequence.append(predicted_word_idx) # Add the predicted word index to the sequence

    return seed_text + ' ' + ' '.join(generated_text)

# Define seed text and number of words to generate
# Modified seed_text to include the comma after 'مهجورة' to match the vocabulary.
seed_text = "مُنذ زمن بعيد كانت هناك قرية قديمة مهجورة،"
num_gen_words = 20

In [30]:
print("\n--- RNN Generated Text ---")
rnn_generated = generate_text(rnn_model, seed_text, num_gen_words, SEQ_LEN)
print(rnn_generated)


--- RNN Generated Text ---
مُنذ زمن بعيد كانت هناك قرية قديمة مهجورة، حيث كانت مليئة بالمنازل والشوارع والمحلات التجاريّة القديمة الفارغة، ممّا جعل هذه القرية مكانًا جيِّدًا لتعيش فيه الفئران! كانت الفئران


In [31]:
print("\n--- LSTM Generated Text ---")
lstm_generated = generate_text(lstm_model, seed_text, num_gen_words, SEQ_LEN)
print(lstm_generated)


--- LSTM Generated Text ---
مُنذ زمن بعيد كانت هناك قرية قديمة مهجورة، حيث كانت مليئة بالمنازل والشوارع والمحلات التجاريّة القديمة الفارغة، ممّا جعل هذه القرية مكانًا جيِّدًا لتعيش فيه الفئران! كانت الفئران


In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

# Get predictions from both models
rnn_predictions = np.argmax(rnn_model.predict(X, verbose=0), axis=-1)
lstm_predictions = np.argmax(lstm_model.predict(X, verbose=0), axis=-1)

# True labels for evaluation
y_true_labels = y

# Calculate metrics for RNN
rnn_accuracy = accuracy_score(y_true_labels, rnn_predictions)
rnn_precision = precision_score(y_true_labels, rnn_predictions, average='weighted', zero_division=0)
rnn_recall = recall_score(y_true_labels, rnn_predictions, average='weighted', zero_division=0)
rnn_f1 = f1_score(y_true_labels, rnn_predictions, average='weighted', zero_division=0)

# Calculate metrics for LSTM
lstm_accuracy = accuracy_score(y_true_labels, lstm_predictions)
lstm_precision = precision_score(y_true_labels, lstm_predictions, average='weighted', zero_division=0)
lstm_recall = recall_score(y_true_labels, lstm_predictions, average='weighted', zero_division=0)
lstm_f1 = f1_score(y_true_labels, lstm_predictions, average='weighted', zero_division=0)

# Calculate diversity between RNN and LSTM predictions
diversity = np.mean(rnn_predictions != lstm_predictions)

# ==========================================
# Final Comparison Table
# ==========================================

comparison = pd.DataFrame({
    'Model': ['RNN', 'LSTM'],
    'Accuracy': [
        rnn_accuracy,
        lstm_accuracy
    ],
    'Precision': [
        rnn_precision,
        lstm_precision
    ],
    'Recall': [
        rnn_recall,
        lstm_recall
    ],
    'F1-Score': [
        rnn_f1,
        lstm_f1
    ]
})

print(comparison)
print("\nDiversity Between RNN and LSTM Predictions:")
print(diversity)


  Model  Accuracy  Precision  Recall  F1-Score
0   RNN       1.0        1.0     1.0       1.0
1  LSTM       1.0        1.0     1.0       1.0

Diversity Between RNN and LSTM Predictions:
0.0
